This run applies four AI-detection tools to the current Testing Sample folders and writes results to a CSV plus a TXT backup. Folders processed: Arabic_AI_Free, Arabic_AI_Generated, Arabic_Humanized, Code_AI_Free, Code_AI_Generated, Code_Humanized. Columns saved (only those relevant to this sample): Language, Label, file_name, word_count, gptzero_ai_rate_percent, gptzero_result, pangram_ai_rate_percent, pangram_result, sapling_ai_rate_percent, sapling_result, isgen_ai_rate_percent, isgen_result. Note: ‘‘AI rate percent’’ and ‘‘result’’ columns correspond to each detector (gptzero, pangram, sapling, isgen). This run reports detector outputs for the current sample only; no additional datasets or derived metrics are produced.

## Before you run

Fill in your API keys in the configuration section.

The notebook:
1. Reads all `.txt` files from the six Testing Sample subfolders.
2. Assigns each file a `Language` and `Label` from its folder.
3. Sends the text to GPTZero, Pangram, Sapling, and Isgen.
4. Saves the results after each file so you can resume safely if the run stops.

In [ ]:
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

BASE_TESTING_SAMPLE_DIR = Path(os.environ["PROJECT_DATA_PATH"])  # requires PROJECT_DATA_PATH in environment

FOLDERS = {
    "Arabic_AI_Free": BASE_TESTING_SAMPLE_DIR
    / "Arabic Assignments"
    / "Trimmed AI-Free Assignments",
    "Arabic_AI_Generated": BASE_TESTING_SAMPLE_DIR
    / "Arabic Assignments"
    / "AI-Generated Assignments",
    "Arabic_Humanized": BASE_TESTING_SAMPLE_DIR
    / "Arabic Assignments"
    / "Humanized AI Assignments",
    "Code_AI_Free": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "AI-Free Code",
    "Code_AI_Generated": BASE_TESTING_SAMPLE_DIR
    / "Coding Assignments"
    / "AI-Generated Code",
    "Code_Humanized": BASE_TESTING_SAMPLE_DIR
    / "Coding Assignments"
    / "Humanized AI Code",
}

OUTPUT_DIR = BASE_TESTING_SAMPLE_DIR / "New_AI_Detection_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample.csv"
BACKUP_TXT = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_BACKUP.txt"
FAILURE_LOG_TXT = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_FAILURES.txt"

GPTZERO_API_KEY = os.environ["GPTZERO_API_KEY"]
PANGRAM_API_KEY = os.environ["PANGRAM_API_KEY"]
SAPLING_API_KEY = os.environ["SAPLING_API_KEY"]
ISGEN_RAPIDAPI_KEY = os.environ["ISGEN_RAPIDAPI_KEY"]

SLEEP_BETWEEN_CALLS = 2.0
MAX_RETRIES = 3
WAIT_SECONDS_BETWEEN_RETRIES = 10
REQUEST_TIMEOUT = 180
RESUME_IF_OUTPUT_EXISTS = True
MAX_TEXT_CHARS = 50000

FINAL_COLUMNS = [
    "Language",
    "Label",
    "file_name",
    "word_count",
    "gptzero_ai_rate_percent",
    "gptzero_result",
    "pangram_ai_rate_percent",
    "pangram_result",
    "sapling_ai_rate_percent",
    "sapling_result",
    "isgen_ai_rate_percent",
    "isgen_result",
]

NUMERIC_COLUMNS = {
    "word_count",
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
}


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text, flags=re.UNICODE)) if text else 0


def read_txt(path):
    # Attempt common encodings in order; fall back to reading with replacement of invalid bytes.
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (OSError, UnicodeDecodeError):
            continue

    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def clean_text(text):
    if not text:
        return ""
    # Replace NULs, normalize line endings, collapse repeated whitespace and excessive blank lines.
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    if not text or len(text) <= max_chars:
        return text

    # Prefer paragraph or sentence boundaries to avoid arbitrary API input cuts.
    cut = text[:max_chars]
    last_paragraph = cut.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return cut[:last_paragraph].strip()

    last_boundary = max(
        cut.rfind(". "),
        cut.rfind("! "),
        cut.rfind("? "),
        cut.rfind("۔ "),
        cut.rfind("\n"),
    )
    if last_boundary > max_chars * 0.6:
        return cut[: last_boundary + 1].strip()
    return cut.strip()


def detect_isgen_lang(language):
    return "ar" if language == "Arabic" else "en"


def request_with_retries(method, url, **kwargs):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.request(method, url, timeout=REQUEST_TIMEOUT, **kwargs)
            if 200 <= response.status_code < 300:
                return response
            last_error = RuntimeError(
                f"HTTP {response.status_code}: {response.text[:1200]}"
            )
        except requests.RequestException as error:
            last_error = error

        if attempt < MAX_RETRIES:
            time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    raise last_error


def call_gptzero(text, api_key):
    response = request_with_retries(
        "POST",
        "https://api.gptzero.me/v2/predict/text",
        headers={
            "x-api-key": api_key,
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "Mozilla/5.0",
        },
        json={"document": text},
    )
    data = response.json()
    document = data.get("documents", [data])[0] if isinstance(data, dict) else None

    if not isinstance(document, dict):
        raise RuntimeError(f"GPTZero unexpected response shape: {str(data)[:500]}")

    probabilities = document.get("class_probabilities", {}) or {}
    return {
        "gptzero_predicted_class": document.get("predicted_class"),
        "gptzero_document_classification": document.get("document_classification"),
        "gptzero_prob_ai": probabilities.get("ai"),
    }


def call_pangram(text, api_key):
    response = request_with_retries(
        "POST",
        "https://text.api.pangram.com/v3",
        headers={"Content-Type": "application/json", "x-api-key": api_key},
        json={"text": text, "public_dashboard_link": False},
    )
    data = response.json()
    return {
        "pangram_prediction_short": data.get("prediction_short"),
        "pangram_prediction": data.get("prediction"),
        "pangram_fraction_ai": data.get("fraction_ai"),
    }


def call_sapling(text, api_key):
    response = request_with_retries(
        "POST",
        "https://api.sapling.ai/api/v1/aidetect",
        json={"key": api_key, "text": text},
    )
    return {"sapling_score": response.json().get("score")}


def call_isgen(text, lang, api_key):
    response = request_with_retries(
        "POST",
        "https://ai-detection4.p.rapidapi.com/v1/ai-detection-rapid-api",
        headers={
            "x-rapidapi-key": api_key,
            "x-rapidapi-host": "ai-detection4.p.rapidapi.com",
            "Content-Type": "application/json",
        },
        json={"text": text, "lang": lang},
    )
    data = response.json()
    summary = data.get("summary", {}) or {}
    return {
        "isgen_ai_score": data.get("ai_score"),
        "isgen_summary_ai": summary.get("ai"),
        "isgen_summary_human": summary.get("human"),
        "isgen_summary_mixed": summary.get("mixed"),
    }


def normalize_gptzero_result(result):
    ai_rate = normalize_percent(result.get("gptzero_prob_ai"))
    label = (
        result.get("gptzero_predicted_class")
        or result.get("gptzero_document_classification")
        or ""
    )
    # If detector didn't provide a label, derive a simple threshold-based label from the probability.
    if not label and pd.notna(ai_rate):
        label = "Likely AI" if ai_rate >= 50 else "Likely Human"
    return ai_rate, str(label)


def normalize_pangram_result(result):
    ai_rate = normalize_percent(result.get("pangram_fraction_ai"))
    label = (
        result.get("pangram_prediction_short") or result.get("pangram_prediction") or ""
    )
    if not label and pd.notna(ai_rate):
        label = "Likely AI" if ai_rate >= 50 else "Likely Human"
    return ai_rate, str(label)


def normalize_sapling_result(result):
    ai_rate = normalize_percent(result.get("sapling_score"))
    label = (
        "Likely AI"
        if pd.notna(ai_rate) and ai_rate >= 50
        else "Likely Human" if pd.notna(ai_rate) else ""
    )
    return ai_rate, label


def normalize_isgen_result(result):
    ai_rate = normalize_percent(result.get("isgen_ai_score"))
    label = (
        "Likely AI"
        if pd.notna(ai_rate) and ai_rate >= 50
        else "Likely Human" if pd.notna(ai_rate) else ""
    )
    return ai_rate, label


def derive_metadata(group_name, txt_path):
    # Map folder group names to human-readable language and label values used in output.
    language = (
        "Arabic"
        if group_name.startswith("Arabic_")
        else "Coding" if group_name.startswith("Code_") else "English"
    )
    label = (
        "AI-Free"
        if group_name.endswith("AI_Free")
        else (
            "AI-Generated"
            if "AI_Generated" in group_name
            else "Humanized AI" if "Humanized" in group_name else "Unknown"
        )
    )
    return {"Language": language, "Label": label, "file_name": txt_path.name}


def load_existing_output():
    if not OUTPUT_CSV.exists():
        return pd.DataFrame(columns=FINAL_COLUMNS)

    output = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
    for column in FINAL_COLUMNS:
        if column not in output.columns:
            output[column] = np.nan if column in NUMERIC_COLUMNS else ""
    return output[FINAL_COLUMNS].copy()


def save_outputs(output):
    output.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    with open(BACKUP_TXT, "w", encoding="utf-8") as file:
        file.write(output.to_string(index=False))


def append_failure_log(message):
    with open(FAILURE_LOG_TXT, "a", encoding="utf-8") as file:
        file.write(f"{message}\n")


def get_existing_row_idx(output, language, file_name, label):
    matches = output.index[
        (output["Language"].astype(str) == str(language))
        & (output["file_name"].astype(str) == str(file_name))
        & (output["Label"].astype(str) == str(label))
    ].tolist()
    return matches[0] if matches else None


def is_row_complete(output, row_idx):
    # Consider a row complete if every required column has a non-empty value (numerics not NaN).
    if row_idx is None or row_idx >= len(output):
        return False

    for column in FINAL_COLUMNS:
        value = output.loc[row_idx, column]
        if column in NUMERIC_COLUMNS:
            if pd.isna(value):
                return False
        elif not str(value).strip():
            return False
    return True


all_jobs = []
for group_name, folder_path in FOLDERS.items():
    if not folder_path.exists():
        print(f"WARNING: missing folder -> {folder_path}")
        continue
    all_jobs.extend(
        (group_name, path)
        for path in sorted(folder_path.rglob("*.txt"))
        if path.is_file()
    )

print(f"Total files found: {len(all_jobs)}")
out = load_existing_output()

for index, (group_name, txt_path) in enumerate(all_jobs, start=1):
    metadata = derive_metadata(group_name, txt_path)
    file_key = f"{metadata['Label']} | {metadata['file_name']}"

    print("=" * 110)
    print(f"[{index}/{len(all_jobs)}] {file_key}")

    row_idx = get_existing_row_idx(
        out, metadata["Language"], metadata["file_name"], metadata["Label"]
    )
    if RESUME_IF_OUTPUT_EXISTS and is_row_complete(out, row_idx):
        print("SKIP: already complete in CSV")
        continue

    if row_idx is None:
        row_idx = len(out)
        for column in FINAL_COLUMNS:
            out.loc[row_idx, column] = np.nan if column in NUMERIC_COLUMNS else ""

    try:
        clean = clean_text(read_txt(txt_path))
        detector_text = truncate_text(clean)
        out.loc[row_idx, "Language"] = metadata["Language"]
        out.loc[row_idx, "Label"] = metadata["Label"]
        out.loc[row_idx, "file_name"] = metadata["file_name"]
        out.loc[row_idx, "word_count"] = count_words(clean)

        detectors = [
            (
                "GPTZero",
                "gptzero",
                call_gptzero,
                (detector_text, GPTZERO_API_KEY),
                normalize_gptzero_result,
            ),
            (
                "Pangram",
                "pangram",
                call_pangram,
                (detector_text, PANGRAM_API_KEY),
                normalize_pangram_result,
            ),
            (
                "Sapling",
                "sapling",
                call_sapling,
                (detector_text, SAPLING_API_KEY),
                normalize_sapling_result,
            ),
            (
                "Isgen",
                "isgen",
                call_isgen,
                (
                    detector_text,
                    detect_isgen_lang(metadata["Language"]),
                    ISGEN_RAPIDAPI_KEY,
                ),
                normalize_isgen_result,
            ),
        ]

        for detector_name, prefix, detector, arguments, normalize in detectors:
            try:
                ai_rate, result_label = normalize(detector(*arguments))
                out.loc[row_idx, f"{prefix}_ai_rate_percent"] = ai_rate
                out.loc[row_idx, f"{prefix}_result"] = result_label
                print(f"{detector_name:<8} -> {ai_rate} | {result_label}")
            except Exception as error:
                out.loc[row_idx, f"{prefix}_result"] = f"ERROR: {str(error)[:200]}"
                append_failure_log(f"{file_key} | {detector_name} | {error}")
                print(f"{detector_name:<8} FAILED -> {error}")
            save_outputs(out)
            time.sleep(SLEEP_BETWEEN_CALLS)

    except Exception as error:
        append_failure_log(f"{file_key} | FILE LEVEL ERROR | {error}")
        print(f"FILE FAILED -> {error}")
        save_outputs(out)

save_outputs(out)

print("\n" + "=" * 110)
print("DONE")
print("=" * 110)
print(f"Rows in output: {len(out)}")
print(f"CSV saved to   : {OUTPUT_CSV}")
print(f"TXT saved to   : {BACKUP_TXT}")
print(f"Failures log   : {FAILURE_LOG_TXT}")

display(out.head(10))

In [ ]:
!pip -q install requests pandas numpy openpyxl

import os
import re
import time
from pathlib import Path, PurePosixPath
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import requests

PROJECT_DATA_PATH = os.environ.get("PROJECT_DATA_PATH")
PANGRAM_API_KEY = os.environ.get("PANGRAM_API_KEY")

if not PROJECT_DATA_PATH:
    raise EnvironmentError("Set PROJECT_DATA_PATH to the testing-sample directory.")
if not PANGRAM_API_KEY:
    raise EnvironmentError("Set PANGRAM_API_KEY before running this cell.")

BASE_TESTING_SAMPLE_DIR = Path(PROJECT_DATA_PATH)

FOLDERS = {
    "Code_AI_Free": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "AI-Free Code",
    "Code_AI_Generated": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "AI-Generated Code",
    "Code_Humanized": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "Humanized AI Code",
}

OUTPUT_DIR = Path(
    os.environ.get(
        "AI_DETECTION_OUTPUT_DIR",
        BASE_TESTING_SAMPLE_DIR / "New_AI_Detection_Results",
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY.csv"
BACKUP_TXT = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY_BACKUP.txt"
FAILURE_LOG_TXT = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY_FAILURES.txt"

PANGRAM_API_URL = "https://text.api.pangram.com/v3"

START_INDEX = 153
END_INDEX = 300
MAX_RETRIES = 4
WAIT_SECONDS_BETWEEN_RETRIES = 10
REQUEST_TIMEOUT = 180
SKIP_ALREADY_FILLED = True
MAX_CODE_WORDS_PER_CHUNK = 2500
SECONDS_BETWEEN_PANGRAM_REQUESTS = 1.0

SEPARATOR_MARKERS = {"############", "##########", "==========", "-----", "__________"}

KNOWN_CODE_EXTS = {
    ".js", ".jsx", ".ts", ".tsx", ".json", ".css", ".scss", ".sass", ".less",
    ".html", ".htm", ".vue", ".md", ".xml", ".yml", ".yaml", ".graphql", ".gql",
    ".sql", ".mjs", ".cjs", ".py", ".java", ".cpp", ".c", ".h", ".hpp",
    ".cs", ".php", ".rb", ".go", ".rs", ".kt", ".swift", ".sh", ".ipynb",
}

FINAL_COLUMNS = [
    "Language",
    "Label",
    "file_name",
    "word_count",
    "gptzero_ai_rate_percent",
    "gptzero_result",
    "pangram_ai_rate_percent",
    "pangram_result",
    "sapling_ai_rate_percent",
    "sapling_result",
    "isgen_ai_rate_percent",
    "isgen_result",
]

NUMERIC_COLUMNS = {
    "word_count",
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
}


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text, flags=re.UNICODE)) if text else 0


def read_txt(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (UnicodeDecodeError, OSError):
            pass

    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def derive_metadata(group_name, txt_path):
    if group_name.endswith("AI_Free"):
        label = "AI-Free"
    elif "AI_Generated" in group_name:
        label = "AI-Generated"
    elif "Humanized" in group_name:
        label = "Humanized AI"
    else:
        label = "Unknown"

    return {"Language": "Coding", "Label": label, "file_name": txt_path.name}


def load_existing_output():
    if not OUTPUT_CSV.exists():
        return pd.DataFrame(columns=FINAL_COLUMNS)

    df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
    for column in FINAL_COLUMNS:
        if column not in df.columns:
            df[column] = np.nan if column in NUMERIC_COLUMNS else ""
    return df[FINAL_COLUMNS].copy()


def save_outputs(df):
    output = df.copy()
    output.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    with open(BACKUP_TXT, "w", encoding="utf-8") as file:
        file.write(output.to_string(index=False))


def append_failure_log(message):
    with open(FAILURE_LOG_TXT, "a", encoding="utf-8") as file:
        file.write(f"{message}\n")


def get_existing_row_idx(df, file_name, label):
    matches = df.index[
        (df["file_name"].astype(str) == str(file_name))
        & (df["Label"].astype(str) == str(label))
    ].tolist()
    return matches[0] if matches else None


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def sanitize_rel_path(rel_path: str) -> str:
    parts = []
    for part in PurePosixPath(rel_path.replace("\\", "/").strip()).parts:
        if part in ("", ".", "..", "/"):
            continue
        cleaned = re.sub(r'[<>:"|?*]', "_", part).replace("\x00", "").strip()
        if cleaned:
            parts.append(cleaned)
    return "/".join(parts)[:240] or "main.js"


def looks_like_path_line(text: str) -> bool:
    text = text.strip()
    if not text or text in SEPARATOR_MARKERS or text.startswith(("http://", "https://")):
        return False
    if len(text) > 260 or not any(text.lower().endswith(ext) for ext in KNOWN_CODE_EXTS):
        return False

    code_tokens = [
        "import ", "from ", "const ", "let ", "var ", "function ", "class ",
        "<", ">", "{", "}", ";", "=>", "(", ")", "=", "require(",
    ]
    return not any(token in text for token in code_tokens) and bool(
        re.fullmatch(r"[A-Za-z0-9_\-./\\ @()\[\]{}+]+", text)
    )


def is_probable_header(lines: List[str], index: int) -> bool:
    if not looks_like_path_line(lines[index]):
        return False

    if index == 0:
        return True

    for previous_index in range(index - 1, -1, -1):
        previous = lines[previous_index].strip()
        if previous:
            return previous in SEPARATOR_MARKERS
    return False


def parse_internal_code_files(text: str) -> List[Dict[str, Any]]:
    lines = normalize_text(text).splitlines()
    files = []
    current_path = None
    current_lines = []
    found_header = False
    skip_one_blank_after_header = False

    def flush_current():
        nonlocal current_path, current_lines
        if current_path is not None:
            content = "\n".join(current_lines).strip()
            if content:
                files.append({
                    "path": current_path,
                    "content": content,
                    "word_count": count_words(content),
                })
        current_path = None
        current_lines = []

    for index, line in enumerate(lines):
        stripped = line.strip()
        if stripped in SEPARATOR_MARKERS:
            continue

        if is_probable_header(lines, index):
            found_header = True
            flush_current()
            current_path = sanitize_rel_path(stripped)
            skip_one_blank_after_header = True
            continue

        if not found_header:
            current_lines.append(line)
            continue

        if skip_one_blank_after_header and not stripped:
            skip_one_blank_after_header = False
            continue

        skip_one_blank_after_header = False
        current_lines.append(line)

    flush_current()

    if not files:
        content = normalize_text(text)
        if content:
            files.append({"path": "main.txt", "content": content, "word_count": count_words(content)})

    return files


def split_code_by_line_word_limit(text: str, max_words: int = MAX_CODE_WORDS_PER_CHUNK) -> List[str]:
    chunks = []
    current_lines = []
    current_words = 0

    for line in normalize_text(text).splitlines():
        line_words = count_words(line)
        if current_lines and current_words + line_words > max_words:
            chunks.append("\n".join(current_lines).strip())
            current_lines = [line]
            current_words = line_words
        else:
            current_lines.append(line)
            current_words += line_words

    if current_lines:
        chunks.append("\n".join(current_lines).strip())

    return [chunk for chunk in chunks if chunk]


def call_pangram_v3(text: str) -> Dict[str, Any]:
    headers = {
        "Content-Type": "application/json",
        "x-api-key": PANGRAM_API_KEY,
        "Accept": "application/json",
    }
    payload = {"text": text, "public_dashboard_link": False}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                PANGRAM_API_URL,
                headers=headers,
                json=payload,
                timeout=REQUEST_TIMEOUT,
            )
            if response.status_code in (429, 500, 502, 503, 504):
                wait_seconds = min(60, WAIT_SECONDS_BETWEEN_RETRIES * attempt)
                print(f"    Attempt {attempt}: HTTP {response.status_code}, retrying in {wait_seconds}s")
                time.sleep(wait_seconds)
                continue
            if response.status_code >= 400:
                raise RuntimeError(f"HTTP {response.status_code}: {response.text[:1000]}")
            return response.json()
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                wait_seconds = min(60, WAIT_SECONDS_BETWEEN_RETRIES * attempt)
                print(f"    Attempt {attempt} failed: {error}")
                time.sleep(wait_seconds)
            else:
                raise RuntimeError(f"All retries failed. Last error: {last_error}") from error


def aggregate_chunk_results(
    chunk_results: List[Dict[str, Any]], chunk_word_counts: List[int]
) -> Dict[str, Any]:
    total_weight = sum(max(1, count) for count in chunk_word_counts)
    fraction_ai = 0.0
    fraction_ai_assisted = 0.0
    fraction_human = 0.0

    # Weight chunk-level estimates by source length before assigning a submission label.
    for result, word_count in zip(chunk_results, chunk_word_counts):
        weight = max(1, word_count)
        fraction_ai += float(result.get("fraction_ai", 0) or 0) * weight
        fraction_ai_assisted += float(result.get("fraction_ai_assisted", 0) or 0) * weight
        fraction_human += float(result.get("fraction_human", 0) or 0) * weight

    if total_weight:
        fraction_ai /= total_weight
        fraction_ai_assisted /= total_weight
        fraction_human /= total_weight

    scores = [("AI", fraction_ai), ("AI-Assisted", fraction_ai_assisted), ("Human", fraction_human)]
    dominant_class = max(scores, key=lambda item: item[1])[0]
    if max(score for _, score in scores) < 0.60:
        dominant_class = "Mixed"

    return {
        "fraction_ai": fraction_ai,
        "fraction_ai_assisted": fraction_ai_assisted,
        "fraction_human": fraction_human,
        "dominant_class": dominant_class,
    }


def run_pangram_code_submission(raw_text: str) -> Dict[str, Any]:
    internal_files = parse_internal_code_files(raw_text)
    if not internal_files:
        raise RuntimeError("No internal code files were parsed from this submission.")

    chunk_results = []
    chunk_word_counts = []

    for file_index, internal_file in enumerate(internal_files, 1):
        chunks = split_code_by_line_word_limit(internal_file["content"])
        print(
            f"    Internal file {file_index}/{len(internal_files)}: {internal_file['path']} "
            f"| words={internal_file['word_count']} | chunks={len(chunks)}"
        )

        for chunk_index, chunk in enumerate(chunks, 1):
            chunk_word_count = count_words(chunk)
            print(f"      Chunk {chunk_index}/{len(chunks)} | words={chunk_word_count}")
            chunk_results.append(call_pangram_v3(chunk))
            chunk_word_counts.append(chunk_word_count)
            time.sleep(SECONDS_BETWEEN_PANGRAM_REQUESTS)

    aggregate = aggregate_chunk_results(chunk_results, chunk_word_counts)
    result = aggregate["dominant_class"]
    result_labels = {
        "AI": "Likely AI",
        "Human": "Likely Human",
        "AI-Assisted": "Mixed / AI-Assisted",
    }

    return {
        "pangram_ai_rate_percent": normalize_percent(aggregate["fraction_ai"]),
        "pangram_result": result_labels.get(result, result or "Unknown"),
    }


coding_jobs = []
for group_name, folder_path in FOLDERS.items():
    if not folder_path.exists():
        print(f"WARNING: missing folder -> {folder_path}")
        continue
    coding_jobs.extend((group_name, path) for path in sorted(folder_path.rglob("*.txt")) if path.is_file())

print(f"Total coding files found: {len(coding_jobs)}")

indexed_jobs = [
    (index, group_name, path)
    for index, (group_name, path) in enumerate(coding_jobs, start=151)
]
jobs_to_run = [job for job in indexed_jobs if START_INDEX <= job[0] <= END_INDEX]
print(f"Jobs selected for this run: {len(jobs_to_run)} | indices {START_INDEX} to {END_INDEX}")

out = load_existing_output()

for global_index, group_name, txt_path in jobs_to_run:
    metadata = derive_metadata(group_name, txt_path)
    file_key = f"{metadata['Label']} | {metadata['file_name']}"

    print("=" * 110)
    print(f"[{global_index}/300] {file_key}")

    row_index = get_existing_row_idx(out, metadata["file_name"], metadata["Label"])
    if row_index is None:
        row_index = len(out)
        for column in FINAL_COLUMNS:
            out.loc[row_index, column] = np.nan if column in NUMERIC_COLUMNS else ""

    for column in ("Language", "Label", "file_name"):
        out.loc[row_index, column] = metadata[column]

    # This run records Pangram results only; inactive detector fields remain blank.
    for column in ("gptzero_ai_rate_percent", "sapling_ai_rate_percent", "isgen_ai_rate_percent"):
        out.loc[row_index, column] = np.nan
    for column in ("gptzero_result", "sapling_result", "isgen_result"):
        out.loc[row_index, column] = ""

    raw_text = read_txt(txt_path)
    out.loc[row_index, "word_count"] = count_words(raw_text)

    pangram_already_filled = (
        pd.notna(out.loc[row_index, "pangram_ai_rate_percent"])
        and str(out.loc[row_index, "pangram_result"]).strip() != ""
    )
    if SKIP_ALREADY_FILLED and pangram_already_filled:
        print("Pangram already filled for this coding row, skipping.")
        save_outputs(out)
        continue

    try:
        pangram_output = run_pangram_code_submission(raw_text)
        out.loc[row_index, "pangram_ai_rate_percent"] = pangram_output["pangram_ai_rate_percent"]
        out.loc[row_index, "pangram_result"] = pangram_output["pangram_result"]
        print(f"Pangram -> {pangram_output['pangram_ai_rate_percent']} | {pangram_output['pangram_result']}")
    except Exception as error:
        print(f"Pangram FAILED -> {error}")
        append_failure_log(f"Pangram failed | index={global_index} | {file_key} | {error}")

    save_outputs(out)

out = out[FINAL_COLUMNS].copy()
save_outputs(out)

print("=" * 110)
print("DONE")
print("=" * 110)
print(f"CSV saved to:    {OUTPUT_CSV}")
print(f"TXT saved to:    {BACKUP_TXT}")
print(f"Failures saved:  {FAILURE_LOG_TXT}")
print("=" * 110)

In [ ]:
!pip -q install requests pandas numpy openpyxl

import os
import re
import time
from pathlib import Path, PurePosixPath
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import requests
from google.colab import drive

DRIVE_MOUNT_PATH = os.environ.get("DRIVE_MOUNT_PATH", "/content/drive")
PROJECT_DATA_PATH = os.environ.get("PROJECT_DATA_PATH")
PANGRAM_API_KEY = os.environ.get("PANGRAM_API_KEY")

if not PROJECT_DATA_PATH:
    raise ValueError("Set PROJECT_DATA_PATH to the directory containing the testing sample.")
if not PANGRAM_API_KEY:
    raise ValueError("Set PANGRAM_API_KEY before running this notebook.")

drive.mount(DRIVE_MOUNT_PATH)

BASE_TESTING_SAMPLE_DIR = Path(PROJECT_DATA_PATH)
FOLDERS = {
    "Code_AI_Free": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "AI-Free Code",
    "Code_AI_Generated": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "AI-Generated Code",
    "Code_Humanized": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "Humanized AI Code",
}

OUTPUT_DIR = BASE_TESTING_SAMPLE_DIR / "New_AI_Detection_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY.csv"
BACKUP_TXT = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY_BACKUP.txt"
FAILURE_LOG_TXT = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY_FAILURES.txt"

PANGRAM_API_URL = "https://text.api.pangram.com/v3"
MAX_RETRIES = 4
WAIT_SECONDS_BETWEEN_RETRIES = 10
REQUEST_TIMEOUT = 180
SECONDS_BETWEEN_PANGRAM_REQUESTS = 1.0
MAX_SUBMISSION_CHARS = 120000

RERUN_SPLIT_RANGE_START = 153
RERUN_SPLIT_RANGE_END = 219
CONTINUE_FROM_INDEX = 220
END_INDEX = 300

FINAL_COLUMNS = [
    "Language",
    "Label",
    "file_name",
    "word_count",
    "gptzero_ai_rate_percent",
    "gptzero_result",
    "pangram_ai_rate_percent",
    "pangram_result",
    "sapling_ai_rate_percent",
    "sapling_result",
    "isgen_ai_rate_percent",
    "isgen_result",
]
NUMERIC_COLUMNS = {
    "word_count",
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
}


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text, flags=re.UNICODE)) if text else 0


def read_txt(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (OSError, UnicodeDecodeError):
            pass

    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def clean_text(text):
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_SUBMISSION_CHARS):
    if not text or len(text) <= max_chars:
        return text

    cut = text[:max_chars]
    last_paragraph = cut.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return cut[:last_paragraph].strip()

    last_line = cut.rfind("\n")
    if last_line > max_chars * 0.6:
        return cut[:last_line].strip()

    return cut.strip()


def derive_metadata(group_name, txt_path):
    if group_name.endswith("AI_Free"):
        label = "AI-Free"
    elif "AI_Generated" in group_name:
        label = "AI-Generated"
    elif "Humanized" in group_name:
        label = "Humanized AI"
    else:
        label = "Unknown"

    return {"Language": "Coding", "Label": label, "file_name": txt_path.name}


def load_existing_output():
    if not OUTPUT_CSV.exists():
        return pd.DataFrame(columns=FINAL_COLUMNS)

    dataframe = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
    for column in FINAL_COLUMNS:
        if column not in dataframe.columns:
            dataframe[column] = np.nan if column in NUMERIC_COLUMNS else ""
    return dataframe[FINAL_COLUMNS].copy()


def save_outputs(dataframe):
    output = dataframe.copy()
    output.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    with open(BACKUP_TXT, "w", encoding="utf-8") as file:
        file.write(output.to_string(index=False))


def append_failure_log(message):
    with open(FAILURE_LOG_TXT, "a", encoding="utf-8") as file:
        file.write(f"{message}\n")


def get_existing_row_idx(dataframe, file_name, label):
    matches = dataframe.index[
        (dataframe["file_name"].astype(str) == str(file_name))
        & (dataframe["Label"].astype(str) == str(label))
    ].tolist()
    return matches[0] if matches else None


# Historical parsing is retained only to identify files previously split.
SEPARATOR_MARKERS = {"############", "##########", "==========", "-----", "__________"}
KNOWN_CODE_EXTS = {
    ".js", ".jsx", ".ts", ".tsx", ".json", ".css", ".scss", ".sass", ".less",
    ".html", ".htm", ".vue", ".md", ".xml", ".yml", ".yaml", ".graphql", ".gql",
    ".sql", ".mjs", ".cjs", ".py", ".java", ".cpp", ".c", ".h", ".hpp",
    ".cs", ".php", ".rb", ".go", ".rs", ".kt", ".swift", ".sh", ".ipynb",
}


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def sanitize_rel_path(rel_path: str) -> str:
    parts = []
    for part in PurePosixPath(rel_path.replace("\\", "/").strip()).parts:
        if part in ("", ".", ".."):
            continue
        clean_part = re.sub(r'[<>:"|?*]', "_", part).replace("\x00", "").strip()
        if clean_part:
            parts.append(clean_part)

    cleaned_path = "/".join(parts)
    return cleaned_path[:240] if cleaned_path else "main.js"


def looks_like_path_line(line: str) -> bool:
    line = line.strip()
    if not line or line in SEPARATOR_MARKERS or line.startswith(("http://", "https://")):
        return False
    if len(line) > 260 or not any(line.lower().endswith(ext) for ext in KNOWN_CODE_EXTS):
        return False

    code_tokens = [
        "import ", "from ", "const ", "let ", "var ", "function ", "class ",
        "<", ">", "{", "}", ";", "=>", "(", ")", "=", "require(",
    ]
    if any(token in line for token in code_tokens):
        return False

    return bool(re.fullmatch(r"[A-Za-z0-9_\-./\\ @()\[\]{}+]+", line))


def is_probable_header(lines: List[str], index: int) -> bool:
    if not looks_like_path_line(lines[index].strip()):
        return False

    previous_nonempty = next((line.strip() for line in reversed(lines[:index]) if line.strip()), None)
    return index == 0 or previous_nonempty in SEPARATOR_MARKERS


def parse_internal_code_files_old(text: str) -> List[Dict[str, Any]]:
    lines = normalize_text(text).splitlines()
    files = []
    current_path = None
    current_lines = []
    found_header = False
    skip_blank_after_header = False

    def flush_current():
        nonlocal current_path, current_lines
        if current_path is not None:
            content = "\n".join(current_lines).strip()
            if content:
                files.append({
                    "path": current_path,
                    "content": content,
                    "word_count": count_words(content),
                })
        current_path = None
        current_lines = []

    for index, line in enumerate(lines):
        stripped = line.strip()
        if stripped in SEPARATOR_MARKERS:
            continue

        if is_probable_header(lines, index):
            found_header = True
            flush_current()
            current_path = sanitize_rel_path(stripped)
            skip_blank_after_header = True
            continue

        if not found_header:
            current_lines.append(line)
            continue

        if skip_blank_after_header and not stripped:
            skip_blank_after_header = False
            continue

        skip_blank_after_header = False
        current_lines.append(line)

    flush_current()

    if not files:
        content = normalize_text(text)
        if content:
            files.append({"path": "main.txt", "content": content, "word_count": count_words(content)})

    return files


def call_pangram_full_submission(text: str) -> Dict[str, Any]:
    headers = {
        "Content-Type": "application/json",
        "x-api-key": PANGRAM_API_KEY,
        "Accept": "application/json",
    }
    payload = {"text": text, "public_dashboard_link": False}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                PANGRAM_API_URL,
                headers=headers,
                json=payload,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code == 401 and "Insufficient credits" in response.text:
                raise RuntimeError(f"HTTP 401: {response.text}")

            if response.status_code in (429, 500, 502, 503, 504):
                wait_seconds = min(60, WAIT_SECONDS_BETWEEN_RETRIES * attempt)
                print(f"    Attempt {attempt}: HTTP {response.status_code}, retrying in {wait_seconds}s")
                time.sleep(wait_seconds)
                continue

            if response.status_code >= 400:
                raise RuntimeError(f"HTTP {response.status_code}: {response.text[:1000]}")

            return response.json()

        except Exception as error:
            last_error = error
            if "Insufficient credits" in str(error):
                raise
            if attempt < MAX_RETRIES:
                wait_seconds = min(60, WAIT_SECONDS_BETWEEN_RETRIES * attempt)
                print(f"    Attempt {attempt} failed: {error}")
                time.sleep(wait_seconds)
            else:
                raise RuntimeError(f"All retries failed. Last error: {last_error}") from error


def normalize_pangram_result(result: Dict[str, Any]):
    ai_rate = normalize_percent(result.get("fraction_ai"))
    scores = {
        "AI": float(result.get("fraction_ai", 0) or 0),
        "AI-Assisted": float(result.get("fraction_ai_assisted", 0) or 0),
        "Human": float(result.get("fraction_human", 0) or 0),
    }
    dominant = max(scores, key=scores.get)
    labels = {
        "AI": "Likely AI",
        "AI-Assisted": "Mixed / AI-Assisted",
        "Human": "Likely Human",
    }
    return ai_rate, labels.get(dominant, "Unknown")


coding_jobs = []
for group_name, folder_path in FOLDERS.items():
    if not folder_path.exists():
        print(f"WARNING: missing folder -> {folder_path}")
        continue

    coding_jobs.extend((group_name, path) for path in sorted(folder_path.rglob("*.txt")) if path.is_file())

indexed_jobs = [
    (index, group_name, txt_path)
    for index, (group_name, txt_path) in enumerate(coding_jobs, start=151)
]
print(f"Total coding files found: {len(indexed_jobs)}")

jobs_to_run = []
for global_index, group_name, txt_path in indexed_jobs:
    internal_file_count = len(parse_internal_code_files_old(read_txt(txt_path)))

    if RERUN_SPLIT_RANGE_START <= global_index <= RERUN_SPLIT_RANGE_END:
        if internal_file_count > 1:
            jobs_to_run.append((global_index, group_name, txt_path, internal_file_count, "rerun_split_file"))
    elif CONTINUE_FROM_INDEX <= global_index <= END_INDEX:
        jobs_to_run.append((global_index, group_name, txt_path, internal_file_count, "continue_after_credits"))

print(f"Jobs selected for this run: {len(jobs_to_run)}")
for job in jobs_to_run[:20]:
    print(job[0], job[2].name, "| old_internal_files =", job[3], "| reason =", job[4])

out = load_existing_output()

for global_index, group_name, txt_path, old_internal_count, rerun_reason in jobs_to_run:
    metadata = derive_metadata(group_name, txt_path)
    file_key = f"{metadata['Label']} | {metadata['file_name']}"

    print("=" * 120)
    print(f"[{global_index}/300] {file_key}")
    print(f"Reason: {rerun_reason} | old_internal_files={old_internal_count}")

    row_index = get_existing_row_idx(out, metadata["file_name"], metadata["Label"])
    if row_index is None:
        row_index = len(out)
        for column in FINAL_COLUMNS:
            out.loc[row_index, column] = np.nan if column in NUMERIC_COLUMNS else ""

    for column, value in metadata.items():
        out.loc[row_index, column] = value

    for detector in ("gptzero", "sapling", "isgen"):
        out.loc[row_index, f"{detector}_ai_rate_percent"] = np.nan
        out.loc[row_index, f"{detector}_result"] = ""

    raw_text = read_txt(txt_path)
    cleaned_text = clean_text(raw_text)
    full_text = truncate_text(cleaned_text)
    out.loc[row_index, "word_count"] = count_words(cleaned_text)

    try:
        print(f"    Sending full submission as one request | words={count_words(full_text)} | chars={len(full_text)}")
        pangram_rate, pangram_result = normalize_pangram_result(call_pangram_full_submission(full_text))
        out.loc[row_index, "pangram_ai_rate_percent"] = pangram_rate
        out.loc[row_index, "pangram_result"] = pangram_result
        print(f"Pangram -> {pangram_rate} | {pangram_result}")

    except Exception as error:
        print(f"Pangram FAILED -> {error}")
        append_failure_log(
            f"Pangram failed | index={global_index} | reason={rerun_reason} | "
            f"old_internal_files={old_internal_count} | {file_key} | {error}"
        )
        if "Insufficient credits" in str(error):
            save_outputs(out)
            raise

    save_outputs(out)
    time.sleep(SECONDS_BETWEEN_PANGRAM_REQUESTS)

out = out[FINAL_COLUMNS].copy()
save_outputs(out)

print("=" * 120)
print("DONE")
print("=" * 120)
print(f"CSV saved to:    {OUTPUT_CSV}")
print(f"TXT saved to:    {BACKUP_TXT}")
print(f"Failures saved:  {FAILURE_LOG_TXT}")
print("=" * 120)

In [ ]:
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

PROJECT_DATA_PATH = os.environ["PROJECT_DATA_PATH"]
PANGRAM_API_KEY = os.environ["PANGRAM_API_KEY"]
PANGRAM_API_URL = "https://text.api.pangram.com/v3"

BASE_TESTING_SAMPLE_DIR = Path(PROJECT_DATA_PATH)
FOLDERS = {
    "Code_AI_Free": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "AI-Free Code",
    "Code_AI_Generated": BASE_TESTING_SAMPLE_DIR
    / "Coding Assignments"
    / "AI-Generated Code",
    "Code_Humanized": BASE_TESTING_SAMPLE_DIR
    / "Coding Assignments"
    / "Humanized AI Code",
}

OUTPUT_DIR = BASE_TESTING_SAMPLE_DIR / "New_AI_Detection_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY.csv"
BACKUP_TXT = (
    OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY_BACKUP.txt"
)
FAILURE_LOG_TXT = (
    OUTPUT_DIR / "AI_Detection_Results_New_Testing_Sample_PANGRAM_ONLY_FAILURES.txt"
)

GLOBAL_INDEX_START = 151
START_INDEX = 220
END_INDEX = 300
MAX_RETRIES = 4
WAIT_SECONDS_BETWEEN_RETRIES = 10
REQUEST_TIMEOUT = 180
SECONDS_BETWEEN_PANGRAM_REQUESTS = 1.0
MAX_SUBMISSION_CHARS = 120000
SKIP_ALREADY_FILLED = True

FINAL_COLUMNS = [
    "Language",
    "Label",
    "file_name",
    "word_count",
    "gptzero_ai_rate_percent",
    "gptzero_result",
    "pangram_ai_rate_percent",
    "pangram_result",
    "sapling_ai_rate_percent",
    "sapling_result",
    "isgen_ai_rate_percent",
    "isgen_result",
]

NUMERIC_COLUMNS = [
    "word_count",
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
]


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except Exception:
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text, flags=re.UNICODE)) if text else 0


def read_txt(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except Exception:
            pass

    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def clean_text(text):
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_SUBMISSION_CHARS):
    if not text or len(text) <= max_chars:
        return text

    # Prefer paragraph or line boundaries when fitting the API character limit.
    cut = text[:max_chars]
    last_paragraph = cut.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return cut[:last_paragraph].strip()

    last_line = cut.rfind("\n")
    if last_line > max_chars * 0.6:
        return cut[:last_line].strip()

    return cut.strip()


def derive_metadata(group_name, txt_path):
    if group_name.endswith("AI_Free"):
        label = "AI-Free"
    elif "AI_Generated" in group_name:
        label = "AI-Generated"
    elif "Humanized" in group_name:
        label = "Humanized AI"
    else:
        label = "Unknown"

    return {"Language": "Coding", "Label": label, "file_name": txt_path.name}


def load_existing_output():
    if not OUTPUT_CSV.exists():
        return pd.DataFrame(columns=FINAL_COLUMNS)

    dataframe = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
    for column in FINAL_COLUMNS:
        if column not in dataframe.columns:
            dataframe[column] = np.nan if column in NUMERIC_COLUMNS else ""
    return dataframe[FINAL_COLUMNS].copy()


def save_outputs(dataframe):
    dataframe.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    with open(BACKUP_TXT, "w", encoding="utf-8") as file:
        file.write(dataframe.to_string(index=False))


def append_failure_log(message):
    with open(FAILURE_LOG_TXT, "a", encoding="utf-8") as file:
        file.write(f"{message}\n")


def get_existing_row_idx(dataframe, file_name, label):
    matches = dataframe.index[
        (dataframe["file_name"].astype(str) == str(file_name))
        & (dataframe["Label"].astype(str) == str(label))
    ].tolist()
    return matches[0] if matches else None


def call_pangram_full_submission(text):
    headers = {
        "Content-Type": "application/json",
        "x-api-key": PANGRAM_API_KEY,
        "Accept": "application/json",
    }
    payload = {"text": text, "public_dashboard_link": False}

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                PANGRAM_API_URL,
                headers=headers,
                json=payload,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code == 401 and "Insufficient credits" in response.text:
                raise RuntimeError(f"HTTP 401: {response.text}")

            if response.status_code in (429, 500, 502, 503, 504):
                wait_seconds = min(60, WAIT_SECONDS_BETWEEN_RETRIES * attempt)
                print(
                    f"    Attempt {attempt}: HTTP {response.status_code}, retrying in {wait_seconds}s"
                )
                time.sleep(wait_seconds)
                continue

            if response.status_code >= 400:
                raise RuntimeError(
                    f"HTTP {response.status_code}: {response.text[:1000]}"
                )

            return response.json()

        except Exception as error:
            last_error = error
            if "Insufficient credits" in str(error):
                raise
            if attempt < MAX_RETRIES:
                wait_seconds = min(60, WAIT_SECONDS_BETWEEN_RETRIES * attempt)
                print(f"    Attempt {attempt} failed: {error}")
                time.sleep(wait_seconds)
            else:
                raise RuntimeError(f"All retries failed. Last error: {last_error}")


def normalize_pangram_result(result):
    ai_rate = normalize_percent(result.get("fraction_ai"))
    ai_assisted = float(result.get("fraction_ai_assisted", 0) or 0)
    human = float(result.get("fraction_human", 0) or 0)
    ai = float(result.get("fraction_ai", 0) or 0)

    dominant = max(
        [("AI", ai), ("AI-Assisted", ai_assisted), ("Human", human)],
        key=lambda item: item[1],
    )[0]

    result_label = {
        "AI": "Likely AI",
        "Human": "Likely Human",
        "AI-Assisted": "Mixed / AI-Assisted",
    }.get(dominant, "Unknown")

    return ai_rate, result_label


coding_jobs = []
for group_name, folder_path in FOLDERS.items():
    if not folder_path.exists():
        print(f"WARNING: missing folder -> {folder_path}")
        continue

    for txt_path in sorted(
        path for path in folder_path.rglob("*.txt") if path.is_file()
    ):
        coding_jobs.append((group_name, txt_path))

indexed_jobs = [
    (index, group_name, txt_path)
    for index, (group_name, txt_path) in enumerate(
        coding_jobs, start=GLOBAL_INDEX_START
    )
]
jobs_to_run = [job for job in indexed_jobs if START_INDEX <= job[0] <= END_INDEX]

print(f"Total coding files found: {len(indexed_jobs)}")
print(
    f"Jobs selected for this run: {len(jobs_to_run)} | indices {START_INDEX} to {END_INDEX}"
)

out = load_existing_output()

for global_idx, group_name, txt_path in jobs_to_run:
    metadata = derive_metadata(group_name, txt_path)
    file_key = f"{metadata['Label']} | {metadata['file_name']}"

    print("=" * 110)
    print(f"[{global_idx}/{END_INDEX}] {file_key}")

    row_idx = get_existing_row_idx(out, metadata["file_name"], metadata["Label"])
    if row_idx is None:
        row_idx = len(out)
        for column in FINAL_COLUMNS:
            out.loc[row_idx, column] = np.nan if column in NUMERIC_COLUMNS else ""

    out.loc[row_idx, "Language"] = metadata["Language"]
    out.loc[row_idx, "Label"] = metadata["Label"]
    out.loc[row_idx, "file_name"] = metadata["file_name"]

    for detector in ("gptzero", "sapling", "isgen"):
        out.loc[row_idx, f"{detector}_ai_rate_percent"] = np.nan
        out.loc[row_idx, f"{detector}_result"] = ""

    cleaned_text = clean_text(read_txt(txt_path))
    full_text = truncate_text(cleaned_text)
    out.loc[row_idx, "word_count"] = count_words(cleaned_text)

    pangram_already_filled = (
        pd.notna(out.loc[row_idx, "pangram_ai_rate_percent"])
        and str(out.loc[row_idx, "pangram_result"]).strip() != ""
    )

    if SKIP_ALREADY_FILLED and pangram_already_filled:
        print("Pangram already filled for this row, skipping.")
        save_outputs(out)
        continue

    try:
        print(
            f"    Sending full submission as one request | "
            f"words={count_words(full_text)} | chars={len(full_text)}"
        )
        pangram_raw = call_pangram_full_submission(full_text)
        pangram_rate, pangram_result = normalize_pangram_result(pangram_raw)

        out.loc[row_idx, "pangram_ai_rate_percent"] = pangram_rate
        out.loc[row_idx, "pangram_result"] = pangram_result
        print(f"Pangram -> {pangram_rate} | {pangram_result}")

    except Exception as error:
        print(f"Pangram FAILED -> {error}")
        append_failure_log(
            f"Pangram failed | index={global_idx} | {file_key} | {error}"
        )
        save_outputs(out)
        if "Insufficient credits" in str(error):
            raise

    save_outputs(out)
    time.sleep(SECONDS_BETWEEN_PANGRAM_REQUESTS)

out = out[FINAL_COLUMNS].copy()
save_outputs(out)

print("=" * 110)
print("DONE")
print("=" * 110)
print(f"CSV saved to:    {OUTPUT_CSV}")
print(f"TXT saved to:    {BACKUP_TXT}")
print(f"Failures saved:  {FAILURE_LOG_TXT}")
print("=" * 110)

In [ ]:
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Configure paths and credentials through environment variables.
DATA_DIR = Path(os.environ["AI_DETECTION_DATA_DIR"])
GPTZERO_API_KEY = os.environ["GPTZERO_API_KEY"]

MERGED_RESULTS_INPUT = Path(
    os.getenv(
        "MERGED_RESULTS_INPUT", DATA_DIR / "AI_Detection_Results_Merged_300_rows.xlsx"
    )
)
MERGED_RESULTS_OUTPUT = Path(
    os.getenv(
        "MERGED_RESULTS_OUTPUT",
        DATA_DIR / "AI_Detection_Results_Merged_300_rows_gptzero.xlsx",
    )
)
REPORTS_RESULTS_INPUT = Path(
    os.getenv(
        "REPORTS_RESULTS_INPUT",
        DATA_DIR / "Second_Year_Reports_AI_Detection_Results.csv",
    )
)
REPORTS_RESULTS_OUTPUT = Path(
    os.getenv(
        "REPORTS_RESULTS_OUTPUT",
        DATA_DIR / "Second_Year_Reports_AI_Detection_Results_gptzero.csv",
    )
)

ARABIC_FOLDERS = {
    "Arabic_AI_Free": Path(os.environ["ARABIC_AI_FREE_DIR"]),
    "Arabic_AI_Generated": Path(os.environ["ARABIC_AI_GENERATED_DIR"]),
    "Arabic_Humanized": Path(os.environ["ARABIC_HUMANIZED_DIR"]),
}
REPORTS_DIR = Path(os.environ["REPORTS_DIR"])

MERGED_BACKUP_TXT = MERGED_RESULTS_OUTPUT.with_suffix(".txt")
REPORTS_BACKUP_TXT = REPORTS_RESULTS_OUTPUT.with_suffix(".txt")
FAILURE_LOG_TXT = Path(os.getenv("FAILURE_LOG_TXT", DATA_DIR / "gptzero_failures.log"))

REQUEST_TIMEOUT = 180
MAX_RETRIES = 4
WAIT_SECONDS_BETWEEN_RETRIES = 12
SLEEP_BETWEEN_CALLS = 2.0
MAX_TEXT_CHARS = 50_000
SKIP_FILLED_ROWS = True


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def read_txt(path: Path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (OSError, UnicodeDecodeError):
            continue

    with path.open("r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    if len(text) <= max_chars:
        return text

    # Prefer a paragraph or line boundary to avoid cutting text mid-structure.
    truncated = text[:max_chars]
    last_paragraph = truncated.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return truncated[:last_paragraph].strip()

    last_line = truncated.rfind("\n")
    if last_line > max_chars * 0.6:
        return truncated[:last_line].strip()

    return truncated.strip()


def append_failure_log(message):
    FAILURE_LOG_TXT.parent.mkdir(parents=True, exist_ok=True)
    with FAILURE_LOG_TXT.open("a", encoding="utf-8") as file:
        file.write(f"{message}\n")


def save_excel_with_txt_backup(df, excel_path, txt_path):
    excel_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(excel_path, index=False)
    with txt_path.open("w", encoding="utf-8") as file:
        file.write(df.to_string(index=False))


def save_csv_with_txt_backup(df, csv_path, txt_path):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    with txt_path.open("w", encoding="utf-8") as file:
        file.write(df.to_string(index=False))


def ensure_gptzero_columns(df):
    if "gptzero_ai_rate_percent" not in df.columns:
        df["gptzero_ai_rate_percent"] = np.nan
    if "gptzero_result" not in df.columns:
        df["gptzero_result"] = ""
    return df


def build_file_index(folders):
    file_map = {}
    for group_name, folder in folders.items():
        if not folder.exists():
            print(f"WARNING: missing folder -> {folder}")
            continue
        for path in sorted(folder.rglob("*.txt")):
            if path.is_file():
                file_map[(path.name, group_name)] = path
    return file_map


def build_reports_index(folder):
    if not folder.exists():
        raise FileNotFoundError(f"Reports folder not found: {folder}")

    return {path.name: path for path in sorted(folder.rglob("*.txt")) if path.is_file()}


def call_gptzero(text):
    url = "https://api.gptzero.me/v2/predict/text"
    headers = {
        "x-api-key": GPTZERO_API_KEY.strip(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                url,
                headers=headers,
                json={"document": text},
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code in (429, 500, 502, 503, 504):
                wait_seconds = WAIT_SECONDS_BETWEEN_RETRIES * attempt
                print(
                    f"    Attempt {attempt}: HTTP {response.status_code}, retrying in {wait_seconds}s"
                )
                time.sleep(wait_seconds)
                continue

            if response.status_code >= 400:
                raise RuntimeError(
                    f"HTTP {response.status_code}: {response.text[:1000]}"
                )

            data = response.json()
            if (
                isinstance(data, dict)
                and isinstance(data.get("documents"), list)
                and data["documents"]
            ):
                document = data["documents"][0]
            else:
                document = data

            if not isinstance(document, dict):
                raise RuntimeError(
                    f"Unexpected GPTZero response shape: {str(data)[:1000]}"
                )

            probabilities = document.get("class_probabilities", {}) or {}
            score = probabilities.get("ai", document.get("completely_generated_prob"))
            score = normalize_percent(score)

            result = str(
                document.get("predicted_class")
                or document.get("document_classification")
                or ""
            )
            if not result and pd.notna(score):
                result = "AI" if float(score) >= 50 else "Human"

            return score, result, data

        except Exception as error:
            last_error = error
            if attempt == MAX_RETRIES:
                raise RuntimeError(
                    f"All retries failed. Last error: {last_error}"
                ) from error

            wait_seconds = WAIT_SECONDS_BETWEEN_RETRIES * attempt
            print(f"    Attempt {attempt} failed: {error}")
            time.sleep(wait_seconds)


if not MERGED_RESULTS_INPUT.exists():
    raise FileNotFoundError(f"Merged results input not found: {MERGED_RESULTS_INPUT}")

merged_df = ensure_gptzero_columns(pd.read_excel(MERGED_RESULTS_INPUT))
arabic_file_index = build_file_index(ARABIC_FOLDERS)

print("=" * 90)
print("Filling GPTZero fields for Arabic rows in merged results")
print("=" * 90)

arabic_mask = merged_df["Language"].astype(str).str.strip().eq("Arabic")
arabic_rows = merged_df[arabic_mask]
print(f"Arabic rows found: {len(arabic_rows)}")

label_groups = {
    "AI-Free": "Arabic_AI_Free",
    "AI-Generated": "Arabic_AI_Generated",
    "Humanized AI": "Arabic_Humanized",
}

for idx in arabic_rows.index:
    file_name = str(merged_df.loc[idx, "file_name"]).strip()
    label = str(merged_df.loc[idx, "Label"]).strip()
    print(f"[MERGED row {idx}] {label} | {file_name}")

    already_filled = (
        pd.notna(merged_df.loc[idx, "gptzero_ai_rate_percent"])
        and str(merged_df.loc[idx, "gptzero_result"]).strip()
    )
    if SKIP_FILLED_ROWS and already_filled:
        print("    GPTZero already filled, skipping.")
        continue

    group_name = label_groups.get(label)
    if group_name is None:
        append_failure_log(f"MERGED | Unknown Arabic label | {label} | {file_name}")
        print("    Unknown Arabic label, skipped.")
        continue

    text_path = arabic_file_index.get((file_name, group_name))
    if text_path is None:
        append_failure_log(f"MERGED | Missing source txt | {group_name} | {file_name}")
        print("    Source txt file not found, skipped.")
        continue

    try:
        text = truncate_text(clean_text(read_txt(text_path)))
        score, result, _ = call_gptzero(text)
        merged_df.loc[idx, "gptzero_ai_rate_percent"] = score
        merged_df.loc[idx, "gptzero_result"] = result
        print(f"    GPTZero -> {score} | {result}")
    except Exception as error:
        append_failure_log(
            f"MERGED | GPTZero failed | {group_name} | {file_name} | {error}"
        )
        print(f"    FAILED -> {error}")

    # Checkpoint after each attempted row to preserve completed requests.
    save_excel_with_txt_backup(merged_df, MERGED_RESULTS_OUTPUT, MERGED_BACKUP_TXT)
    time.sleep(SLEEP_BETWEEN_CALLS)

save_excel_with_txt_backup(merged_df, MERGED_RESULTS_OUTPUT, MERGED_BACKUP_TXT)

if not REPORTS_RESULTS_INPUT.exists():
    raise FileNotFoundError(f"Reports results CSV not found: {REPORTS_RESULTS_INPUT}")

reports_df = ensure_gptzero_columns(
    pd.read_csv(REPORTS_RESULTS_INPUT, encoding="utf-8-sig")
)
reports_file_index = build_reports_index(REPORTS_DIR)

print("\n" + "=" * 90)
print("Filling GPTZero fields for report rows")
print("=" * 90)
print(f"Report rows found: {len(reports_df)}")

for idx in reports_df.index:
    file_name = str(reports_df.loc[idx, "file_name"]).strip()
    print(f"[REPORTS row {idx}] {file_name}")

    already_filled = (
        pd.notna(reports_df.loc[idx, "gptzero_ai_rate_percent"])
        and str(reports_df.loc[idx, "gptzero_result"]).strip()
    )
    if SKIP_FILLED_ROWS and already_filled:
        print("    GPTZero already filled, skipping.")
        continue

    text_path = reports_file_index.get(file_name)
    if text_path is None:
        append_failure_log(f"REPORTS | Missing source txt | {file_name}")
        print("    Source txt file not found, skipped.")
        continue

    try:
        text = truncate_text(clean_text(read_txt(text_path)))
        score, result, _ = call_gptzero(text)
        reports_df.loc[idx, "gptzero_ai_rate_percent"] = score
        reports_df.loc[idx, "gptzero_result"] = result
        print(f"    GPTZero -> {score} | {result}")
    except Exception as error:
        append_failure_log(f"REPORTS | GPTZero failed | {file_name} | {error}")
        print(f"    FAILED -> {error}")

    save_csv_with_txt_backup(reports_df, REPORTS_RESULTS_OUTPUT, REPORTS_BACKUP_TXT)
    time.sleep(SLEEP_BETWEEN_CALLS)

save_csv_with_txt_backup(reports_df, REPORTS_RESULTS_OUTPUT, REPORTS_BACKUP_TXT)

print("\n" + "=" * 90)
print("Completed")
print("=" * 90)
print(f"Merged output: {MERGED_RESULTS_OUTPUT}")
print(f"Merged backup: {MERGED_BACKUP_TXT}")
print(f"Reports output: {REPORTS_RESULTS_OUTPUT}")
print(f"Reports backup: {REPORTS_BACKUP_TXT}")
print(f"Failure log: {FAILURE_LOG_TXT}")

In [ ]:
!pip -q install gspread gspread-dataframe pandas numpy

import os
import re
from collections import defaultdict

import pandas as pd
import gspread
from google.auth import default
from google.colab import auth
from gspread_dataframe import get_as_dataframe, set_with_dataframe

auth.authenticate_user()

SHEET_NAME = os.environ.get("AI_DETECTION_SHEET_NAME")
MAIN_TAB_NAME = os.environ.get("AI_DETECTION_MAIN_TAB") or None
THRESHOLD = 50.0

if not SHEET_NAME:
    raise ValueError("Set the AI_DETECTION_SHEET_NAME environment variable before running this cell.")

TOOL_CONFIG = {
    "gptzero": {
        "rate_col": "gptzero_ai_rate_percent",
        "result_col": "gptzero_result",
    },
    "pangram": {
        "rate_col": "pangram_ai_rate_percent",
        "result_col": "pangram_result",
    },
    "sapling": {
        "rate_col": "sapling_ai_rate_percent",
        "result_col": "sapling_result",
    },
    "isgen": {
        "rate_col": "isgen_ai_rate_percent",
        "result_col": "isgen_result",
    },
}

creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SHEET_NAME)
ws = sh.sheet1 if MAIN_TAB_NAME is None else sh.worksheet(MAIN_TAB_NAME)

df = get_as_dataframe(ws, evaluate_formulas=False, dtype=str)
df = df.dropna(how="all").copy()
df.columns = [str(column).strip() for column in df.columns]

print(f"Opened Google Sheet: {sh.title}")
print(f"Using tab: {ws.title}")
print(f"Rows: {len(df)}")


def safe_str(value):
    return "" if pd.isna(value) else str(value).strip()


def clean_key(value):
    value = safe_str(value).lower()
    value = re.sub(r"[_\-]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def canonical_label(label):
    value = clean_key(label)
    if value in {"ai free", "aifree", "human", "human written", "human-written", "fully human", "ai-free"}:
        return "AI-Free"
    if value in {"ai generated", "ai-generated", "ai full", "ai-full", "full ai", "fully ai", "pure ai"}:
        return "AI-Generated"
    if value in {"humanized ai", "humanized", "humanized-ai", "ai humanized"}:
        return "Humanized AI"
    return safe_str(label)


def true_class_from_label(label):
    label = canonical_label(label)
    if label == "AI-Free":
        return "Human"
    if label in {"AI-Generated", "Humanized AI"}:
        return "AI"
    return None


def canonical_language(language):
    value = clean_key(language)
    if value in {"arabic", "arabic text", "text", "arabic files"}:
        return "Arabic"
    if value in {"coding", "code", "coding files", "programming"}:
        return "Coding"
    return safe_str(language)


def parse_numeric_score(value):
    if pd.isna(value):
        return None

    value = str(value).strip().replace(",", "")
    if not value:
        return None

    match = re.search(r"-?\d+(\.\d+)?", value)
    if not match:
        return None

    try:
        return float(match.group(0))
    except ValueError:
        return None


def detect_scale_for_tool(dataframe, rate_col):
    """Treat columns with only values at or below 1 as fractional scores."""
    if rate_col not in dataframe.columns:
        return 100.0

    values = [
        score
        for value in dataframe[rate_col].tolist()
        if (score := parse_numeric_score(value)) is not None
    ]
    if not values:
        return 100.0
    return 1.0 if max(values) <= 1.0 else 100.0


def get_ai_score_percent(raw_value, scale_hint):
    score = parse_numeric_score(raw_value)
    if score is None:
        return None
    if scale_hint == 1.0:
        score *= 100.0
    return max(0.0, min(100.0, score))


def binary_prediction_from_result_or_score(result_value, ai_score_percent):
    """Use numeric scores when available; otherwise use recognizable text results."""
    if ai_score_percent is not None:
        return "AI" if ai_score_percent >= THRESHOLD else "Human"

    value = clean_key(result_value)
    if not value:
        return None

    human_phrases = [
        "human",
        "likely human",
        "fully human",
        "human written",
        "human-written",
        "predominantly human",
    ]
    ai_phrases = [
        "ai",
        "likely ai",
        "ai generated",
        "generated by ai",
        "ai written",
        "ai-written",
        "machine generated",
        "fully ai",
    ]

    if any(phrase in value for phrase in human_phrases) and not any(
        phrase in value for phrase in ["likely ai", "ai generated", "generated by ai"]
    ):
        return "Human"
    if any(phrase in value for phrase in ai_phrases) and "human" not in value:
        return "AI"
    return None


def init_stats():
    return {
        "n_rows": 0,
        "n_binary": 0,
        "n_soft": 0,
        "n_ai_truth": 0,
        "n_human_truth": 0,
        "tp": 0,
        "fp": 0,
        "tn": 0,
        "fn": 0,
        "sum_soft_correct": 0.0,
        "sum_ai_score_all": 0.0,
        "sum_ai_score_on_ai_truth": 0.0,
        "sum_ai_score_on_human_truth": 0.0,
        "n_ai_score_all": 0,
        "n_ai_score_on_ai_truth": 0,
        "n_ai_score_on_human_truth": 0,
    }


def safe_div(numerator, denominator):
    return None if denominator == 0 else numerator / denominator


def finalize_stats(group_dict):
    rows = []
    for key, stats in group_dict.items():
        if len(key) == 2:
            language, tool = key
            label = None
        else:
            language, label, tool = key

        tp, fp, tn, fn = (stats[name] for name in ("tp", "fp", "tn", "fn"))
        binary_accuracy = safe_div(tp + tn, tp + fp + tn + fn)
        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        specificity = safe_div(tn, tn + fp)
        f1 = None if precision is None or recall is None or precision + recall == 0 else 2 * precision * recall / (precision + recall)
        balanced_accuracy = None if recall is None or specificity is None else (recall + specificity) / 2
        soft_accuracy = safe_div(stats["sum_soft_correct"], stats["n_soft"])

        row = {
            "Language": language,
            "Tool": tool,
            "N_Rows_With_GroundTruth": stats["n_rows"],
            "N_Binary_Evaluable": stats["n_binary"],
            "N_Soft_Evaluable": stats["n_soft"],
            "N_True_AI": stats["n_ai_truth"],
            "N_True_Human": stats["n_human_truth"],
            "TP": tp,
            "FP": fp,
            "TN": tn,
            "FN": fn,
            "Binary_Accuracy_50_Percent": None if binary_accuracy is None else round(binary_accuracy * 100, 4),
            "Precision_AI_Percent": None if precision is None else round(precision * 100, 4),
            "Recall_AI_Percent": None if recall is None else round(recall * 100, 4),
            "Specificity_Human_Percent": None if specificity is None else round(specificity * 100, 4),
            "F1_AI_Percent": None if f1 is None else round(f1 * 100, 4),
            "Balanced_Accuracy_Percent": None if balanced_accuracy is None else round(balanced_accuracy * 100, 4),
            "Soft_Accuracy_Percent": None if soft_accuracy is None else round(soft_accuracy * 100, 4),
            "Mean_AI_Score_All_Percent": None if stats["n_ai_score_all"] == 0 else round(stats["sum_ai_score_all"] / stats["n_ai_score_all"], 4),
            "Mean_AI_Score_On_True_AI_Percent": None if stats["n_ai_score_on_ai_truth"] == 0 else round(stats["sum_ai_score_on_ai_truth"] / stats["n_ai_score_on_ai_truth"], 4),
            "Mean_AI_Score_On_True_Human_Percent": None if stats["n_ai_score_on_human_truth"] == 0 else round(stats["sum_ai_score_on_human_truth"] / stats["n_ai_score_on_human_truth"], 4),
        }
        if label is not None:
            row = {"Language": language, "Label": label, **{key: value for key, value in row.items() if key != "Language"}}
        rows.append(row)
    return pd.DataFrame(rows)


def replace_sheet(sheet_title, output_df):
    try:
        sh.del_worksheet(sh.worksheet(sheet_title))
    except gspread.exceptions.WorksheetNotFound:
        pass

    worksheet = sh.add_worksheet(
        title=sheet_title,
        rows=max(len(output_df) + 10, 20),
        cols=max(len(output_df.columns) + 5, 10),
    )
    set_with_dataframe(worksheet, output_df, include_index=False, include_column_header=True, resize=True)
    return worksheet


required_columns = ["Language", "Label"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}. Found: {list(df.columns)}")

tool_scale = {
    tool: detect_scale_for_tool(df, config["rate_col"])
    for tool, config in TOOL_CONFIG.items()
}

print("Detected score scales:")
for tool, scale in tool_scale.items():
    description = "0..1 scaled to 0..100" if scale == 1.0 else "0..100"
    print(f"  {tool}: {description}")

overall_stats = defaultdict(init_stats)
label_stats = defaultdict(init_stats)

for tool in TOOL_CONFIG:
    for column in (f"{tool}_pred_label_50", f"{tool}_soft_correctness_percent"):
        if column not in df.columns:
            df[column] = ""

for index, row in df.iterrows():
    language = canonical_language(row.get("Language"))
    label = canonical_label(row.get("Label"))
    truth = true_class_from_label(label)

    if truth is None:
        continue

    for tool, config in TOOL_CONFIG.items():
        rate_col = config["rate_col"]
        result_col = config["result_col"]
        if rate_col not in df.columns and result_col not in df.columns:
            continue

        raw_score = row.get(rate_col) if rate_col in df.columns else None
        raw_result = row.get(result_col) if result_col in df.columns else None
        ai_score = get_ai_score_percent(raw_score, tool_scale[tool]) if rate_col in df.columns else None
        prediction = binary_prediction_from_result_or_score(raw_result, ai_score)

        stat_groups = (
            overall_stats[(language, tool)],
            label_stats[(language, label, tool)],
        )

        for stats in stat_groups:
            stats["n_rows"] += 1
            stats["n_ai_truth" if truth == "AI" else "n_human_truth"] += 1

        if prediction is not None:
            for stats in stat_groups:
                stats["n_binary"] += 1
                if prediction == "AI" and truth == "AI":
                    stats["tp"] += 1
                elif prediction == "AI":
                    stats["fp"] += 1
                elif truth == "Human":
                    stats["tn"] += 1
                else:
                    stats["fn"] += 1
            df.at[index, f"{tool}_pred_label_50"] = prediction

        if ai_score is None:
            continue

        # Soft correctness preserves score magnitude rather than binarizing it.
        soft_correct = ai_score / 100 if truth == "AI" else (100 - ai_score) / 100
        for stats in stat_groups:
            stats["n_soft"] += 1
            stats["sum_soft_correct"] += soft_correct
            stats["sum_ai_score_all"] += ai_score
            stats["n_ai_score_all"] += 1
            if truth == "AI":
                stats["sum_ai_score_on_ai_truth"] += ai_score
                stats["n_ai_score_on_ai_truth"] += 1
            else:
                stats["sum_ai_score_on_human_truth"] += ai_score
                stats["n_ai_score_on_human_truth"] += 1

        df.at[index, f"{tool}_soft_correctness_percent"] = round(soft_correct * 100, 4)

overall_columns = [
    "Language", "Tool", "N_Rows_With_GroundTruth", "N_Binary_Evaluable", "N_Soft_Evaluable",
    "N_True_AI", "N_True_Human", "TP", "FP", "TN", "FN",
    "Binary_Accuracy_50_Percent", "Precision_AI_Percent", "Recall_AI_Percent",
    "Specificity_Human_Percent", "F1_AI_Percent", "Balanced_Accuracy_Percent",
    "Soft_Accuracy_Percent", "Mean_AI_Score_All_Percent",
    "Mean_AI_Score_On_True_AI_Percent", "Mean_AI_Score_On_True_Human_Percent",
]
label_columns = ["Language", "Label", *overall_columns[1:]]

overall_df = finalize_stats(overall_stats).reindex(columns=overall_columns)
label_df = finalize_stats(label_stats).reindex(columns=label_columns)
overall_df = overall_df.sort_values(["Language", "Tool"]).reset_index(drop=True)
label_df = label_df.sort_values(["Language", "Label", "Tool"]).reset_index(drop=True)

overall_lookup = {
    (safe_str(row["Language"]), safe_str(row["Tool"])): row
    for _, row in overall_df.iterrows()
}
soft_accuracy = {}
binary_accuracy = {}

for tool in TOOL_CONFIG:
    row = overall_lookup.get(("Arabic", tool))
    soft_accuracy[tool] = 0.0 if row is None or pd.isna(row["Soft_Accuracy_Percent"]) else float(row["Soft_Accuracy_Percent"]) / 100
    binary_accuracy[tool] = 0.0 if row is None or pd.isna(row["Binary_Accuracy_50_Percent"]) else float(row["Binary_Accuracy_50_Percent"]) / 100

soft_total = sum(soft_accuracy.values())
binary_total = sum(binary_accuracy.values())
soft_weights = {tool: 0.0 if soft_total == 0 else soft_accuracy[tool] / soft_total for tool in TOOL_CONFIG}
binary_weights = {tool: 0.0 if binary_total == 0 else binary_accuracy[tool] / binary_total for tool in TOOL_CONFIG}

arabic_weights_df = pd.DataFrame([
    {
        "Tool": tool,
        "Arabic_Soft_Accuracy_Fraction": round(soft_accuracy[tool], 6),
        "Arabic_Soft_Accuracy_Percent": round(soft_accuracy[tool] * 100, 4),
        "Normalized_Weight_From_Soft_Accuracy": round(soft_weights[tool], 6),
        "Arabic_Binary_Accuracy_50_Fraction": round(binary_accuracy[tool], 6),
        "Arabic_Binary_Accuracy_50_Percent": round(binary_accuracy[tool] * 100, 4),
        "Normalized_Weight_From_Binary_Accuracy_50": round(binary_weights[tool], 6),
    }
    for tool in TOOL_CONFIG
])

for tool in TOOL_CONFIG:
    for column in (f"{tool}_weight_arabic_soft", f"{tool}_weight_arabic_binary50"):
        if column not in df.columns:
            df[column] = ""

arabic_mask = df["Language"].apply(lambda value: canonical_language(value) == "Arabic")
for tool in TOOL_CONFIG:
    df.loc[arabic_mask, f"{tool}_weight_arabic_soft"] = round(soft_weights[tool], 6)
    df.loc[arabic_mask, f"{tool}_weight_arabic_binary50"] = round(binary_weights[tool], 6)

definitions_df = pd.DataFrame([
    ["Positive class", "AI"],
    ["Binary threshold", f"Predicted AI if AI score >= {THRESHOLD}"],
    ["TP", "Predicted AI and true label is AI"],
    ["FP", "Predicted AI and true label is Human / AI-Free"],
    ["TN", "Predicted Human and true label is Human / AI-Free"],
    ["FN", "Predicted Human and true label is AI"],
    ["Binary_Accuracy_50_Percent", "(TP + TN) / (TP + FP + TN + FN) * 100"],
    ["Precision_AI_Percent", "TP / (TP + FP) * 100"],
    ["Recall_AI_Percent", "TP / (TP + FN) * 100"],
    ["Specificity_Human_Percent", "TN / (TN + FP) * 100"],
    ["F1_AI_Percent", "2 * Precision * Recall / (Precision + Recall)"],
    ["Balanced_Accuracy_Percent", "(Recall + Specificity) / 2 * 100"],
    ["Soft_Accuracy_Percent", "Average per-row soft correctness * 100"],
    ["Per-row soft correctness if true AI", "AI_score / 100"],
    ["Per-row soft correctness if true Human", "(100 - AI_score) / 100"],
    ["Arabic soft weights", "Normalized from Arabic soft accuracy across configured tools"],
    ["Arabic binary weights", "Normalized from Arabic binary accuracy across configured tools"],
], columns=["Metric", "Definition"])

ws.clear()
set_with_dataframe(ws, df, include_index=False, include_column_header=True, resize=True)
replace_sheet("Overall_Tool_Metrics", overall_df)
replace_sheet("Metrics_By_Label", label_df)
replace_sheet("Arabic_Tool_Weights", arabic_weights_df)
replace_sheet("Metric_Definitions", definitions_df)

print(f"Completed writeback: {sh.url}")